In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").is_dir() and (candidate / "project.ipynb").is_file():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pyspark.sql import SparkSession
from src.dataOI import DataOI

SEED = 42

# True: rebuild inputs and the selected embeddings, then save those results.
# False: skip that work and load the saved train, validation, and test embedding frames from data/processed/embeddings.
REBUILD = False

# Override model hyperparameters here; an empty dictionary uses the defaults.
para = {}

# `spark.driver.memory` is read only when the JVM starts. Restart the kernel after changing it.
spark = (
    SparkSession.builder 
    .master("local[*]")
    .appName("DrugReviewClassification")
    .config("spark.driver.memory", "4g")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark_context = spark.sparkContext
spark_context.setLogLevel("ERROR")
data_oi = DataOI()


In [ ]:
from src.cleaning import data_cleaning
from src.text_features import text_feature_engineering
from src.encoder import encoder
from src.split import split_and_weight

if REBUILD:
    df_raw = data_oi.load_raw_data()
    df_cleaned = data_cleaning(df_raw)
    df_cleaned.cache().count()
    df_engineered = text_feature_engineering(df_cleaned,True)
    df_engineered.cache().count()
    df_train, df_valid, df_test = split_and_weight(df_engineered, SEED)
    df_train, df_valid, df_test = encoder(df_train, df_valid, df_test)
    data_oi.save_split_data(df_train, df_valid, df_test)


In [ ]:
from src.embeddings import EmbeddingModel
from src.text_features import fit_phrases

USE_EMBEDDINGS = {"tfidf": True, "glove": False}
embedding_names = [name for name, enabled in USE_EMBEDDINGS.items() if enabled]

train_embedding_dict = {}
valid_embedding_dict = {}
test_embedding_dict = {}

if REBUILD:
    phrases = fit_phrases(df_train) if "glove" in embedding_names else None
    embedding_model = EmbeddingModel(df_train, df_valid, df_test, phrases)
    for name in embedding_names:
        print(f"Embedding Model {name} Training")
        EmbeddingModel.train_registry[name](embedding_model)
        train_embedding_dict[name], valid_embedding_dict[name], test_embedding_dict[name] = (
            EmbeddingModel.predict_registry[name](embedding_model)
        )
        data_oi.save_embedding_data(
            name,
            train_embedding_dict[name],
            valid_embedding_dict[name],
            test_embedding_dict[name],
        )
else:
    for name in embedding_names:
        train_embedding_dict[name], valid_embedding_dict[name], test_embedding_dict[name] = (
            data_oi.load_embedding_data(name)
        )

print(f"Embeddings in this run: {embedding_names}")


In [ ]:
from src.analytics import AnalyticsModel

analytics_name = "random_forest"
analytics_models = {
    name: AnalyticsModel(
        name,
        train_embedding_dict[name],
        valid_embedding_dict[name],
        test_embedding_dict[name],
    )
    for name in embedding_names
}


In [ ]:
train_pred_dict = {}
valid_pred_dict = {}
test_pred_dict = {}

for embedding_name in embedding_names:
    print(f"Embedding Model {embedding_name} Analytics Model {analytics_name} Running")
    model_name = f"{analytics_name}__{embedding_name}"
    train_pred_dict[model_name], valid_pred_dict[model_name], test_pred_dict[model_name] = (
        AnalyticsModel.registry[analytics_name](
            analytics_models[embedding_name], para=para
        )
    )
    data_oi.save_prediction_data(
        model_name,
        train_pred_dict[model_name],
        valid_pred_dict[model_name],
        test_pred_dict[model_name],
    )


In [ ]:
from src.evaluation import Scorer

# Score only predictions produced by this experiment.
scorer_train = Scorer(train_pred_dict)
print("Training Data:")
scorer_train.show_metrics(truncate = 80)

scorer_valid = Scorer(valid_pred_dict)
print("Validation Data:")
scorer_valid.show_metrics(truncate = 80)

scorer_test = Scorer(test_pred_dict)
print("Testing Data:")
scorer_test.show_metrics(truncate = 80)